## Futásidő-összevetés: rprev referencia (naiv K-szoros) vs. kiterjesztett több indexdátumos implementáció

Ez a notebook a szakdolgozat 5. és 7. fejezetében rögzített több indexdátumos kiterjesztéshez kapcsolódó **futásidő-mérési protokollt** valósítja meg. A mérés célja annak kvantifikálása, hogy azonos bemeneti adaton és azonos szimulációs beállítások mellett milyen számítási igénye van  
(i) a referencia-megoldásnak, ahol a $\{t_k\}_{k=1}^K$ rácsot **$K$ külön `prevalence()` függvényhívással** állítjuk elő, illetve  
(ii) a kiterjesztett rprev-implementációnak, ahol az `index` argumentum több időpont fogadására is képes, és a becslés **egyetlen eljárásfuttatásban** állítja elő a teljes prevalenciavektort és annak összefoglalóját.

### Kapcsolódás a dolgozathoz

A kiterjesztés matematikai tartalma röviden: adott $t_1 < \cdots < t_K$ rácson, az $r$-edik Monte Carlo futásban a futáson belül előállított incidens esetek $(D_{i\ell}^{(r)}, \mathbf{x}_{i\ell}^{(r)})$ és a futáson belül rögzített túlélési függvény $S^{(r)}(\cdot \mid \mathbf{x})$ alapján minden $t_k$ időpontra képezzük a prevalens státuszt, majd időpontonként összegezve kapjuk a $P^{(r)}(t_k)$ prevalenciakomponenseket és a futások feletti összefoglaló statisztikákat.

Csomagszinten a notebook azt a működést tekinti „kiterjesztett” megvalósításnak, ahol a `prevalence()` az `index` bemenetet rendezett $\texttt{index\_dates} = (t_1,\ldots,t_K)$ vektorrá normalizálja, a belső szimulációs réteg futásonként az `index_dates` rácson szolgáltat státuszinformációt, amelyből az összegző réteg időpontonként képez prevalencia-összefoglalót. A kimeneti objektum tartalmazza az `index_dates` vektort és az időpontindexelt eredményeket.

### Mérési egységek és protokoll

A dolgozat terminológiáját követve:

- **Alapkiértékelés:** a `prevalence()` egy futtatása rögzített $t$ mellett.
- **Eljárásfuttatás:** a $\{t_k\}_{k=1}^K$ időpontrács teljes kimenetének előállítása.

A naív, referencia-megoldásban egy eljárásfuttatás **$K$ darab alapkiértékelésből** áll (külön-külön időpontra), míg a kiterjesztett, natív módon több indexpontot támogató megoldásban ugyanez **egyetlen alapkiértékelésként** valósul meg.

A notebook minden paraméterkombinációra ismételt futásidő-mérést végez (`reps`), és az időeredményeke `system.time(...)[["elapsed"]]` alapján rögzíti. A riportolt érték minden beállításpontra ezen ismételt futásokból származó eredmények átlaga.

A mérési protokoll kizárólag a futásidő összevetésére szolgál. A prevalenciabecslés pontosságának vizsgálata ettől elkülönítve, egy másik notebookban történik.

### Bemeneti adat és indexdátumok

A méréshez az rprev csomag `prevsim` példaadata szolgál alapul, melyből visszatevéses mintavétellel áll elő az $N$ soros, regiszterjellegű bemeneti tábla. Az indexdátumok egy rögzített kezdődátumtól (`index_start`) fix naplépéssel (`index_step_days`) generált, rendezett $K$ hosszú sorozatot alkotnak.

A notebookban az indexdátumok generálása determinisztikus: adott $K$, `index_start` és `index_step_days` mellett az `index_dates` teljesen reprodukálható, és a különböző mérési pontok között kizárólag a konfigurációs paraméterek változnak.

A bemeneti táblát a futásidő skálázódásának vizsgálata motiválja, ezért az adat-előállítás során nem cél egy valós regiszter teljes, esetszintű konzisztenciájának biztosítása. A visszatevéses mintavétel következtében előfordulhatnak ismétlődő rekordok, ezért a kapott állomány nem tekintendő tiszta regiszternek a szokásos adatminőségi kritériumok szerint. A mérés érvényességét viszont ez nem érinti, mivel a vizsgálat tárgya kizárólag a számítási terhelés és annak $N$ és $K$ szerinti alakulása.

### Szimulációs és bootstrap beállítások

A mérések rácsa a következő paraméterek mentén szervezett:

- $N$: a bemeneti tábla sorainak száma (adatméret, rekordszám),
- $K$: az indexdátumok száma,
- `N_boot`: a `prevalence()` hívásban használt bootstrap ismétlések száma,
- `reps`: az időmérés ismétlésszáma paraméterpontonként.

A fenti rács mellett a hívásokban több beállítás rögzített, és a futtatások között nem változik: az indexdátum-generálás paraméterei, valamint a túlélési eloszlás választása.

### Kimenet

A futás végén egy eredménytábla készül, amely minden paraméterpontra tartalmazza:

- `N`, `K`, `N_boot`,
- `reps`,
- `elapsed_sec_mean`: az eljárásfuttatás átlagos futásideje másodpercben.

---


#### 1. Közös paraméter konfiguráció
- A futásidő-mérés paraméterrácsának és a rögzített modellspecifikációs beállításoknak a központi definiálása a reprodukálhatóság érdekében.
- A referencia és a kiterjesztett megoldása futtatása ugyanebből a paraméterkészletből dolgozik, így az összevetés konzisztens.

In [1]:
# COMMON_CFG <- list(
#   # Méretek / rácsparaméterek
#   N_values              = c(1e2, 1e3),        # bemeneti tábla sorszámai (rekordszámok)
#   K_values              = c(1, 5),            # indexdátumok száma (K időpont)
#   N_boot_values         = c(50, 100),         # bootstrap ismétlések száma
#   reps                  = 3L,                 # ismétlések száma paraméterpontonként (átlagolt idő)
#   seed                  = 20260227L,          # közös seed a reprodukálható futtatáshoz
#   num_years_to_estimate = c(15),              # becslési horizont (év)
#   index_start           = as.Date("2013-01-30"), # első indexdátum
#   index_step_days       = 30L,                # indexdátumok lépésköze napokban
#   dist                  = "weibull",          # túlélési eloszlás típusa
#   population_size       = 1e6                 # populációméret a prevalenciához
# )

COMMON_CFG <- list(
  # Méretek / rácsparaméterek
  N_values              = c(1e3, 1e4),        # bemeneti tábla sorszámai (rekordszámok)
  K_values              = c(1, 10, 20, 30),   # indexdátumok száma (K időpont)
  N_boot_values         = c(100, 500, 1000),  # bootstrap ismétlések száma
  reps                  = 3L,                 # ismétlések száma paraméterpontonként (átlagolt idő)
  seed                  = 20260227L,          # közös seed a reprodukálható futtatáshoz
  num_years_to_estimate = c(15),              # becslési horizont (év)
  index_start           = as.Date("2013-01-30"), # első indexdátum
  index_step_days       = 30L,                # indexdátumok lépésköze napokban
  dist                  = "weibull",          # túlélési eloszlás típusa
  population_size       = 1e6                 # populációméret a prevalenciához
)

#### 2. Környezet előkészítése és skálázott bemenetek képzése
- A szükséges csomagok betöltése után a `prevsim` szintetikus példaadat beolvasása.
- A `prevsim` táblából, a konfigurációban megadott `N_values` értékekre visszatevéses mintavétellel több, kontrollált méretű bemeneti tábla készül, melyek egységes alapot biztosítanak a futásidőmérésekhez.
- A konfigurációban megadott `K_values` értékek alapján indexdátum-sorozat készítése a későbbi futtatások egységes paraméterezéséhez.

In [8]:
# Csomagok betöltése
suppressPackageStartupMessages({
  library(rprev)
  library(survival)
})

# Példaadat betöltése (kiinduló tábla)
data(prevsim, package = "rprev")
base_df <- as.data.frame(prevsim)


In [ ]:
# Skálázott bemenetek előállítása: minden N-hez egy N soros minta (visszatevéses)
scaled_data <- setNames(
  lapply(COMMON_CFG$N_values, function(N) {
    set.seed(as.integer(COMMON_CFG$seed) + as.integer(N))
    base_df[sample.int(nrow(base_df), size = as.integer(N), replace = TRUE), , drop = FALSE]
  }),
  paste0("N", COMMON_CFG$N_values)
)


In [10]:
# Indexdátumok előállítása
index_dates_by_k <- setNames(
  lapply(COMMON_CFG$K_values, function(K) {
    seq.Date(
      from = COMMON_CFG$index_start,
      by = as.integer(COMMON_CFG$index_step_days),
      length.out = as.integer(K)
    )
  }),
  as.character(COMMON_CFG$K_values)
)

index_dates_chr_by_k <- lapply(index_dates_by_k, format, "%Y-%m-%d")


#### 3. Referencia-megoldás futásidő-mérése: K külön `prevalence()` hívással
- A segédfüggvények által előállított $K$ darab indexdátum alapján a referencia-megoldás $K$ külön hívással elvégzi a prevalencia belést.
- Az időmérés `system.time()` alapján történik, és paraméterpontonként a `reps` számú ismétlésből származó futásidők átlagolásra kerülnek.

In [ ]:
# Referencia-megoldás: prevalence() hívás összeállítása
run_rprev_once <- function(dat, index_date_chr, N_boot) {
  rprev::prevalence(
    index = index_date_chr,
    num_years_to_estimate = COMMON_CFG$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = COMMON_CFG$dist,
    population_size = COMMON_CFG$population_size,
    death_column = "eventdate",
    N_boot = N_boot
  )
}

# Referencia-megoldás: Az összeállított prevalence() függvény K külön hívása, és összidő számítása
measure_one_setting <- function(N, K, N_boot) {
  dat <- scaled_data[[paste0("N", as.integer(N))]]

  idx_dates_chr <- index_dates_chr_by_k[[as.character(K)]]

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  elapsed <- numeric(COMMON_CFG$reps)
  for (r in seq_len(COMMON_CFG$reps)) {
    gc()
    t <- system.time({
      for (d_chr in idx_dates_chr) {
        invisible(run_rprev_once(dat, d_chr, N_boot))
      }
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = COMMON_CFG$reps,
    elapsed_sec_mean = mean(elapsed)
  )
}

# Referencia-megoldás: Paraméterrács futtatása és eredménytábla előállítása
set.seed(COMMON_CFG$seed)
results <- do.call(
  rbind,
  lapply(COMMON_CFG$N_values, function(N) {
    do.call(rbind, lapply(COMMON_CFG$K_values, function(K) {
      do.call(rbind, lapply(COMMON_CFG$N_boot_values, function(N_boot) {
        message(sprintf("Run: N=%g, K=%d, N_boot=%d", N, K, N_boot))
        ű_one_setting(
          N = N,
          K = K,
          N_boot = N_boot
        )
      }))
    }))
  })
)

results <- results[order(results$N, results$K, results$N_boot), ]
within(results, { elapsed_sec_mean <- round(elapsed_sec_mean, 2) })

# write.csv(results, "rprev_runtime_pilot.csv", row.names = FALSE)

Run: N=1000, K=1, N_boot=100

Run: N=1000, K=1, N_boot=500

Run: N=1000, K=1, N_boot=1000

Run: N=1000, K=10, N_boot=100

Run: N=1000, K=10, N_boot=500

Run: N=1000, K=10, N_boot=1000

Run: N=1000, K=20, N_boot=100

Run: N=1000, K=20, N_boot=500

Run: N=1000, K=20, N_boot=1000

Run: N=1000, K=30, N_boot=100

Run: N=1000, K=30, N_boot=500

Run: N=1000, K=30, N_boot=1000

Run: N=10000, K=1, N_boot=100

Run: N=10000, K=1, N_boot=500

Run: N=10000, K=1, N_boot=1000

Run: N=10000, K=10, N_boot=100

Run: N=10000, K=10, N_boot=500

Run: N=10000, K=10, N_boot=1000

Run: N=10000, K=20, N_boot=100

Run: N=10000, K=20, N_boot=500

Run: N=10000, K=20, N_boot=1000

Run: N=10000, K=30, N_boot=100

Run: N=10000, K=30, N_boot=500

Run: N=10000, K=30, N_boot=1000



,N,K,N_boot,reps,elapsed_sec_mean
,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,1000,1,100,3,1.34
2,1000,1,500,3,5.16
3,1000,1,1000,3,11.59
4,1000,10,100,3,12.68
5,1000,10,500,3,63.90
6,1000,10,1000,3,144.86
7,1000,20,100,3,32.02
8,1000,20,500,3,139.33
9,1000,20,1000,3,244.44


#### 4. Kiterjesztett rprev csomag betöltése lokális forrásból és git-meta rögzítése
- A csomag lokális forrásból kerül betöltésre, így a futtatás ténylegesen a vizsgált, módosított implementációval történik.
- A futtatott verzió azonosíthatóságát a git ág és commit azonosító kiírása biztosítja.

In [ ]:
# Projekt lokális gyökérkönyvtárának meghatározása (git repo root)
project_root <- trimws(system2("git", c("rev-parse", "--show-toplevel"), stdout = TRUE))
if (length(project_root) == 0 || !dir.exists(project_root)) {
  stop("Could not resolve project root from git.")
}

# Telepített rprev leválasztása, majd lokális forrásból betöltés
if ("package:rprev" %in% search()) detach("package:rprev", unload = TRUE, character.only = TRUE)
if ("rprev" %in% loadedNamespaces()) unloadNamespace("rprev")
if (!requireNamespace("devtools", quietly = TRUE)) stop("devtools package is not available.")

suppressPackageStartupMessages(devtools::load_all(project_root, quiet = TRUE))

# Betöltött csomag elérési útjának megjelenítése
cat(
  "Loaded rprev from: ",
  # normalizePath(getNamespaceInfo("rprev", "path"), winslash = "/"),
  # "\n",
  sep = ""
)

Loaded rprev from: ...Szakdolgozat/repo/rprev-ext

In [ ]:
# Git meta rögzítése: futtatott kódverzió azonosítása
git_cmd <- function(args) {
  out <- suppressWarnings(
    tryCatch(system2("git", args, stdout = TRUE, stderr = TRUE), error = function(e) character(0))
  )
  status <- attr(out, "status")
  if (!is.null(status) || length(out) == 0) return(NA_character_)
  trimws(out[[1]])
}

old_wd <- getwd()                         # munkakönyvtár mentése
on.exit(setwd(old_wd), add = TRUE)        # visszaállítás a cella végén
setwd(project_root)                       # git parancsok futtatása a repo gyökeréből

branch <- git_cmd(c("rev-parse", "--abbrev-ref", "HEAD"))
commit <- git_cmd(c("rev-parse", "HEAD"))

cat("Git branch: ", branch, "\n", sep = "")
cat("Git commit: ", commit, "\n", sep = "")

Git branch: notebooks/runtime-benchmarks
Git commit: 728f7d428ff78092db36b170f1316cb8e4fa21fe


#### 5. Kiterjesztett megoldás futásidő-mérése: natív több indexdátumos `prevalence()` hívás
- Az eljárás egyetlen `prevalence()` hívásban kapja meg a $K$ darab indexdátumot, és ugyanebben a futtatásban állítja elő a teljes kimenetet.
- Az időmérés azonos módon, `system.time()` alapján történik, és `reps` ismétlés átlaga kerül riportolásra.

In [ ]:
# Kiterjesztett megoldás: egy prevalence() hívás több indexdátumra
run_rprev_ext_once <- function(dat, index_dates_chr, N_boot) {
  rprev::prevalence(
    index = index_dates_chr,
    num_years_to_estimate = COMMON_CFG$num_years_to_estimate,
    data = dat,
    inc_formula  = entrydate ~ sex,
    surv_formula = Surv(time, status) ~ age + sex,
    dist = COMMON_CFG$dist,
    population_size = COMMON_CFG$population_size,
    death_column = "eventdate",
    N_boot = N_boot
  )
}

#  Kiterjesztett megoldás: egy hívás futásideje
measure_one_setting_ext <- function(N, K, N_boot) {
  dat <- scaled_data[[paste0("N", as.integer(N))]]

  idx_dates_chr <- index_dates_chr_by_k[[as.character(K)]]

  if (length(idx_dates_chr) != K || anyNA(idx_dates_chr)) {
    stop("Index dátum generálás hibás: idx_dates_chr NA-t tartalmaz vagy rossz hossz.")
  }

  #  Kiterjesztett megoldás: `reps` számú ismétlésből az idő átlaga, majd eredménysor visszaadása
  elapsed <- numeric(COMMON_CFG$reps)
  for (r in seq_len(COMMON_CFG$reps)) {
    gc()
    t <- system.time({
      invisible(run_rprev_ext_once(dat, idx_dates_chr, N_boot))
    })
    elapsed[r] <- unname(t[["elapsed"]])
  }

  data.frame(
    N = N,
    K = K,
    N_boot = N_boot,
    reps = COMMON_CFG$reps,
    elapsed_sec_mean = mean(elapsed)
  )
}

#  Kiterjesztett megoldás: Paraméterrács futtatása és eredménytábla előállítása
set.seed(COMMON_CFG$seed)
results_ext <- do.call(
  rbind,
  lapply(COMMON_CFG$N_values, function(N) {
    do.call(rbind, lapply(COMMON_CFG$K_values, function(K) {
      do.call(rbind, lapply(COMMON_CFG$N_boot_values, function(N_boot) {
        message(sprintf("Ext run: N=%g, K=%d, N_boot=%d", N, K, N_boot))
        measure_one_setting_ext(
          N = N,
          K = K,
          N_boot = N_boot
        )
      }))
    }))
  })
)

results_ext <- results_ext[order(results_ext$N, results_ext$K, results_ext$N_boot), ]
within(results_ext, { elapsed_sec_mean <- round(elapsed_sec_mean, 2) })

Ext run: N=1000, K=1, N_boot=100

Ext run: N=1000, K=1, N_boot=500

Ext run: N=1000, K=1, N_boot=1000

Ext run: N=1000, K=10, N_boot=100

Ext run: N=1000, K=10, N_boot=500

Ext run: N=1000, K=10, N_boot=1000

Ext run: N=1000, K=20, N_boot=100

Ext run: N=1000, K=20, N_boot=500

Ext run: N=1000, K=20, N_boot=1000

Ext run: N=1000, K=30, N_boot=100

Ext run: N=1000, K=30, N_boot=500

Ext run: N=1000, K=30, N_boot=1000

Ext run: N=10000, K=1, N_boot=100

Ext run: N=10000, K=1, N_boot=500

Ext run: N=10000, K=1, N_boot=1000

Ext run: N=10000, K=10, N_boot=100

Ext run: N=10000, K=10, N_boot=500

Ext run: N=10000, K=10, N_boot=1000

Ext run: N=10000, K=20, N_boot=100

Ext run: N=10000, K=20, N_boot=500

Ext run: N=10000, K=20, N_boot=1000

Ext run: N=10000, K=30, N_boot=100

Ext run: N=10000, K=30, N_boot=500

Ext run: N=10000, K=30, N_boot=1000



,N,K,N_boot,reps,elapsed_sec_mean
,<dbl>,<dbl>,<dbl>,<int>,<dbl>
1,1000,1,100,3,2.31
2,1000,1,500,3,10.39
3,1000,1,1000,3,21.83
4,1000,10,100,3,6.78
5,1000,10,500,3,33.19
6,1000,10,1000,3,64.85
7,1000,20,100,3,12.19
8,1000,20,500,3,57.83
9,1000,20,1000,3,116.81


#### 6. Összesítés: referencia- és kiterjesztett futásidők összevetése
- A két eredménytábla paraméterpontonként összeillesztésre kerül, így közvetlenül összehasonlítható a referencia- és a kiterjesztett megoldás átlagos futásideje.
- A különbség abszolút (`delta_sec`) és relatív (`delta_pct`) formában is kiszámításra kerül.

In [ ]:
# Összesítés: referencia- és több indexdátumos futásidők összevezetése és különbségek számítása
summary_runtime <- merge(
  results[, c("N", "K", "N_boot", "reps", "elapsed_sec_mean")],
  results_ext[, c("N", "K", "N_boot", "reps", "elapsed_sec_mean")],
  by = c("N", "K", "N_boot", "reps"),
  suffixes = c("_ref", "_multi")
)

summary_runtime <- summary_runtime[order(summary_runtime$N, summary_runtime$K, summary_runtime$N_boot), ]

# Abszolút és relatív eltérés (több indexdátum - referencia)
summary_runtime$delta_sec <- summary_runtime$elapsed_sec_mean_multi - summary_runtime$elapsed_sec_mean_ref
summary_runtime$delta_pct <- 100 * summary_runtime$delta_sec / summary_runtime$elapsed_sec_mean_ref

# Megjelenítés: kerekítés a táblázatos riporthoz
summary_runtime_disp <- within(summary_runtime, {
  elapsed_sec_mean_ref <- round(elapsed_sec_mean_ref, 2)
  elapsed_sec_mean_multi <- round(elapsed_sec_mean_multi, 2)
  delta_sec <- round(delta_sec, 2)
  delta_pct <- round(delta_pct, 2)
})

summary_runtime_disp

,N,K,N_boot,reps,elapsed_sec_mean_ref,elapsed_sec_mean_multi,delta_sec,delta_pct
,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
1,1000,1,100,3,1.34,2.31,0.97,72.82
3,1000,1,500,3,5.16,10.39,5.23,101.29
2,1000,1,1000,3,11.59,21.83,10.24,88.38
4,1000,10,100,3,12.68,6.78,-5.90,-46.54
6,1000,10,500,3,63.90,33.19,-30.71,-48.06
5,1000,10,1000,3,144.86,64.85,-80.01,-55.23
7,1000,20,100,3,32.02,12.19,-19.84,-61.94
9,1000,20,500,3,139.33,57.83,-81.50,-58.49
8,1000,20,1000,3,244.44,116.81,-127.62,-52.21
